# lora-stack-lab Kaggle runner
Set `BRANCH`/`REPO`, enable **Internet** in notebook settings, run **Save & Run All (Commit)**.
Outputs: `results/` JSONs + `ckpt/` (small) + `results_pack.zip` for manual download.

In [ ]:
import os, subprocess, sys

BRANCH  = "main"                # <- change per stage
STAGE   = "stage0"               # <- stage prefix of configs/experiments/ (stage1, stage2-*)
REPO    = "P-Bao/lora-stack-lab"  # public repo
COMMIT  = None                    # <- optional: pin exact commit for reproduction
RUN_IDS = []                      # <- empty = run whole stage from configs/experiments/

sh = lambda c: subprocess.run(c, shell=True, check=True)
sh("rm -rf /tmp/repo")  # idempotent: always clone fresh (stale /tmp/repo = stale code)
sh(f"git clone --depth 1 --branch {BRANCH} https://github.com/{REPO}.git /tmp/repo")
if COMMIT:
    sh(f"cd /tmp/repo && git fetch --depth 1 origin {COMMIT} && git checkout {COMMIT}")
os.chdir("/tmp/repo")
sh("bash scripts/bootstrap_kaggle.sh")
sys.path.insert(0, "/tmp/repo/src")

In [ ]:
ids = " ".join(RUN_IDS)
sh(f"python scripts/run_queue.py --stage {STAGE} --ids {ids} "
   f"--out /kaggle/working/results --ckpt /kaggle/working/ckpt "
   f"--skip-from /kaggle/input")  # scans previous outputs mounted as input to skip done IDs

In [ ]:
sh("python scripts/collect_results.py --src /kaggle/working/results --zip /kaggle/working/results_pack.zip")